In [119]:
import nltk
import numpy as np
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

nltk.download("punkt_tab")
nltk.download("wordnet")


[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ML\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\ML\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [120]:
# ===============================
# 1. Preprocessing (NO Stopwords)
# ===============================
def preprocess(text):
    tokens = word_tokenize(text.lower())
    
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word.isalpha()]
    
    return tokens

In [121]:
# ===============================
# 2. Vocabulary
# ===============================
def build_vocab(tokens):
    vocab = list(set(tokens))
    
    word2idx = {w: i for i, w in enumerate(vocab)}
    idx2word = {i: w for w, i in word2idx.items()}
    
    return vocab, word2idx, idx2word


In [122]:

# ===============================
# 3. One Hot Encoding
# ===============================
def one_hot(word, word2idx):
    vec = np.zeros(len(word2idx))
    vec[word2idx[word]] = 1
    return vec

In [123]:
# ===============================
# 4. Create CBOW Data
# ===============================
def create_data(tokens, word2idx, window=2):
    X, Y = [], []
    
    for i in range(window, len(tokens) - window):
        context = []
        
        for j in range(i - window, i + window + 1):
            if j != i:
                vec = np.zeros(len(word2idx))
                vec[word2idx[tokens[j]]] = 1
                context.append(vec)
        
        target = np.zeros(len(word2idx))
        target[word2idx[tokens[i]]] = 1
        
        X.append(np.mean(context, axis=0))
        Y.append(target)
    
    return np.array(X), np.array(Y)


In [124]:
# ===============================
# 5. Softmax
# ===============================
def softmax(x):
    e = np.exp(x - np.max(x))
    return e / np.sum(e)


# ===============================
# 6. Forward Propagation
# ===============================
def forward(x, W1, W2):
    h = np.dot(x, W1)          # (vocab_size → embed_dim)
    u = np.dot(h, W2)          # (embed_dim → vocab_size)
    y_pred = softmax(u)        # OUTPUT must be vocab_size
    
    return y_pred, h


# ===============================
# 7. Backward Propagation
# ===============================
def backward(x, h, y, t, W2):

    error = y - t

    dW2 = np.outer(h, error)
    dW1 = np.outer(x, np.dot(W2, error))

    return dW1, dW2


In [125]:
# ===============================
# 8. Train CBOW
# ===============================
def train(X, Y, vocab_size, embed_dim=10, lr=0.01, epochs=200):
    
    W1 = np.random.rand(vocab_size, embed_dim)
    W2 = np.random.rand(embed_dim, vocab_size)

    for epoch in range(epochs):
        loss = 0
        
        for x, y in zip(X, Y):
            y_pred, h = forward(x, W1, W2)
            error = y_pred - y
            
            dW2 = np.outer(h, error)
            dW1 = np.outer(x, np.dot(W2, error))
            
            W1 -= lr * dW1
            W2 -= lr * dW2
            
            loss += -np.sum(y * np.log(y_pred + 1e-9))
        
        if epoch % 50 == 0:
            print(f"Epoch {epoch}, Loss: {loss:.4f}")
    
    return W1, W2


In [126]:
# ===============================
# 9. Get Word Embeddings
# ===============================
def get_embeddings(W1, vocab):

    embeddings = {}
    for i, word in enumerate(vocab):
        embeddings[word] = W1[i]

    return embeddings


# ===============================
# 10. Prediction
# ===============================
def predict(context_words, W1, W2, word2idx, idx2word):
    
    context_vecs = [one_hot(w, word2idx) for w in context_words if w in word2idx]
    x = np.mean(context_vecs, axis=0)
    
    y_pred, _ = forward(x, W1, W2)
    
    return idx2word[np.argmax(y_pred)]


In [127]:
def main():

    corpus = """
    Flower are beautiful and colorful.
    Roses are red and smell very nice.
    Sunflowers are yellow and grow tall.
    Plants need water and sunlight to grow.
    Gardeners take care of flowers and plants.
    Bees visit flowers to collect nectar.
    Flowers bloom in spring season.
    Nature looks fresh with blooming flowers.
    """

    # Preprocess
    tokens = preprocess(corpus)
    print("Tokens:", tokens)

    # Vocabulary
    vocab, word2idx, idx2word = build_vocab(tokens)
  

    X, Y = create_data(tokens, word2idx)
    W1, W2 = train(X, Y, len(vocab))

    # Get embeddings
    embeddings = get_embeddings(W1, vocab)

    print("\nWord Embedding Example:")
    print("flower →", embeddings["flower"])

    # Test prediction (simple + meaningful)
    test = ["flower", "are", "beautiful", "colorful"]

    result = predict(test, W1, W2, word2idx, idx2word)

    print("\nContext:", test)
    print("Predicted Word:", result)


# ===============================
# Run
# ===============================
if __name__ == "__main__":
    main()

Tokens: ['flower', 'are', 'beautiful', 'and', 'colorful', 'rose', 'are', 'red', 'and', 'smell', 'very', 'nice', 'sunflower', 'are', 'yellow', 'and', 'grow', 'tall', 'plant', 'need', 'water', 'and', 'sunlight', 'to', 'grow', 'gardener', 'take', 'care', 'of', 'flower', 'and', 'plant', 'bee', 'visit', 'flower', 'to', 'collect', 'nectar', 'flower', 'bloom', 'in', 'spring', 'season', 'nature', 'look', 'fresh', 'with', 'blooming', 'flower']
Epoch 0, Loss: 163.9216
Epoch 50, Loss: 145.9521
Epoch 100, Loss: 132.7697
Epoch 150, Loss: 113.8317

Word Embedding Example:
flower → [-0.14437848  0.2571298  -0.05609581  1.40476417  0.36522995  0.17068893
 -1.52826525  1.5048735   1.50778731  0.35233192]

Context: ['flower', 'are', 'beautiful', 'colorful']
Predicted Word: and
